# NNLM (Neural Network Language Model)

In [39]:
import torch
import torch.nn as nn
import torch.optim as optim


In [40]:
# 컨텍스트 단어들을 임베딩 -> MLP로 변환해서 다음 단어 분포를 예측하는 모델
class NNLM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, context_size):
        super(NNLM, self).__init__()  # nn.Module 초기화
        self.embed = nn.Embedding(vocab_size, embed_size) # 단어 ID -> 임베딩 벡터
        self.fc1 = nn.Linear(context_size * embed_size, hidden_size) # (컨텍스트* 임베딩 연결) -> 은닉층
        self.relu = nn.ReLU() # 비선형 추가
        self.fc2 = nn.Linear(hidden_size, vocab_size) # 은닉층 -> 단어 분포 logit(어휘 크기)
        self.log_softmax = nn.LogSoftmax(dim=1) # logit -> 로그 확률(배치 기준 dim = 1 배치별 확률)
    
    def forward(self, x):
        embeds = self.embed(x)                   # (B, context_size) -> (B, context_size, embed_size)
        embeds = embeds.view(embeds.size(0), -1) # (B, context_size * embed_size)로 연결 컨텍스트*임베딩 연결
        output = self.fc1(embeds)                # 은닉층 선형 변환
        output = self.relu(output)               # ReLU로 비선형 추가
        output = self.fc2(output)                # 은닉층 -> 어휘 크기만큼 로짓 출력
        log_probs = self.log_softmax(output)     # (B, vocab_size)로 로그 확률 반환
        return log_probs

In [41]:
# 하이퍼파라미터 설정
VOCAB_SIZE = 5000
EMBED_SIZE = 300
HIDDEN_SIZE = 128
CONTEXT_SIZE = 2

model = NNLM(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, CONTEXT_SIZE)
model

NNLM(
  (embed): Embedding(5000, 300)
  (fc1): Linear(in_features=600, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=5000, bias=True)
  (log_softmax): LogSoftmax(dim=1)
)

In [42]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                   Param #
NNLM                                     --
├─Embedding: 1-1                         1,500,000
├─Linear: 1-2                            76,928
├─ReLU: 1-3                              --
├─Linear: 1-4                            645,000
├─LogSoftmax: 1-5                        --
Total params: 2,221,928
Trainable params: 2,221,928
Non-trainable params: 0

In [43]:
# 더미 데이터 생성
X = torch.randint(0, VOCAB_SIZE, (8, CONTEXT_SIZE)) # 배치 8 , 컨텍스트 2 범위로 단어 ID 랜덤 생성
y = torch.randint(0, VOCAB_SIZE, (8,))              # 배치 8 다음단어 ID 랜덤 생성

X.shape, y.shape

(torch.Size([8, 2]), torch.Size([8]))

In [44]:
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

model.train()
optimizer.zero_grad()
output = model(X)
loss = criterion(output, y)
loss.backward()
optimizer.step()

loss.item()

8.478487014770508

ngram 기반의 간단한 텍스트 생성기

In [45]:
import nltk
from nltk.util import ngrams
from collections import Counter
import random

# seed 단어에서 시작해서 bigram 빈도 기반으로 다음 단어를 확률적으로 뽑아 문장 생성
def generate_text_bigram(seed, unigram_freq, bigram_freq, max_len=10):
    current_word = seed        # 첫 단어를 현재 단어로 선택
    generated = [current_word] # 생성 결과 리스트

    for _ in range(max_len - 1): # 최대 길이만큼 반복 (이미 seed 1개 포함)
        # 현재 단어로 시작하는 bigram만 후보군으로 선정
        candidates = [
            (bigram, freq)
            for bigram, freq in bigram_freq.items()
            if bigram[0] == current_word
        ]

        if not candidates: # 후보가 없으면 종료
            break

        # 다음 단어와 빈도를 따로 뽑아서 리스트 분리
        words, freqs = zip(*[
            (bigram[1], freq)
            for bigram, freq in candidates
        ])

        total = sum(freqs) # 빈도 전체 합
        probs = [f / total for f in freqs] # 빈도를 확률로 변환

        next_word = random.choices(words, weights=probs)[0]
        generated.append(next_word)
        current_word = next_word

    return " ".join(generated)

words, freqs = zip(*[(bigram[1], freq) for bigram, freq in candidates])
- bigram : ('자연어', '처리') / bigram[1] => '처리'
- zip : 같은 위치끼리 묶음
- * : unpacking (리스트들을 풀어서 zip에 넣어서 활용)

In [46]:
train_text = "자연어 처리는 재미있다. 자연어 처리는 어렵지만 도전하고 싶다. 오늘은 날씨가 좋다"

train_tokens = nltk.word_tokenize(train_text)

unigram = train_tokens
bigrams = list(ngrams(train_tokens, 2))

unigram_freq = Counter(unigram)
bigram_freq = Counter(bigrams)

In [47]:
generate_text_bigram("자연어", unigram_freq, bigram_freq, max_len = 1000)

'자연어 처리는 재미있다 . 자연어 처리는 재미있다 . 오늘은 날씨가 좋다'